# Piano con las manos
Proyecto de Ciencia de Datos. Modelo preentrenado: **MediaPipe Hands** (21 landmarks por mano).

- **Modo cancion:** la pantalla te dice que mano y que dedo bajar; al bajarlo suena la nota. Incluye *Jingle Bells* completa.
- **Celebracion:** al terminar una cancion hay confeti, fanfarria y un resumen con tu tiempo y fallos.
- **Modo libre:** cada uno de los 10 dedos toca una nota; toca lo que quieras.
- Usa **las dos manos**: los 10 dedos son 10 notas consecutivas.

**Teclas en la ventana de la camara:** `1-5` elegir cancion, `L` modo libre, `R` reiniciar, `C` calibrar con las manos abiertas, `,` `.` octava (modo libre), `D` datos, `Q` salir.


## 1. Librerias

In [1]:
import random
import time

import cv2
import mediapipe as mp
import numpy as np
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


## 2. Configuracion y canciones
Aqui se cambia la camara, la sensibilidad (`FACTOR_ABAJO`, `FACTOR_ARRIBA`) y agregar canciones nuevas a la lista `CANCIONES` (nota + octava, por ejemplo `E5`; solo notas blancas y que quepan en 10 notas consecutivas).

In [2]:
# ----------------------------- CONFIGURACION -----------------------------
CAMARA = 0
SUAVIZADO = 0.5        # EMA sobre la flexion (0 = sin suavizar)
FACTOR_ABAJO = 0.70    # dedo "abajo" si su flexion < 70 % de la mano abierta
FACTOR_ARRIBA = 0.82   # dedo "arriba" de nuevo si supera el 82 %
COOLDOWN = 0.10        # segundos minimos entre dos toques del mismo dedo

# dedo -> (nombre, landmark de la punta)
DEDOS = {1: ("Pulgar", 4), 2: ("Indice", 8), 3: ("Medio", 12),
         4: ("Anular", 16), 5: ("Menique", 20)}
# Flexion tipica con la mano abierta (la tecla C la recalibra)
BASE_ABIERTA = {1: 1.7, 2: 2.0, 3: 2.0, 4: 1.95, 5: 1.8}

NOMBRES = ["Do", "Re", "Mi", "Fa", "Sol", "La", "Si"]
LETRAS = "CDEFGAB"
SEMITONOS = [0, 2, 4, 5, 7, 9, 11]

# Canciones: nota + octava (C4 = Do central). Solo notas blancas (Do mayor).
# Para agregar una cancion, anade otra tupla (nombre, "notas ...").
# Debe caber en 10 notas consecutivas (10 dedos).
JINGLE_BELLS = (
    # estrofa
    "G4 E5 D5 C5 G4 G4 G4 G4 E5 D5 C5 A4  "
    "A4 F5 E5 D5 B4 G5 G5 F5 D5 E5  "
    "G4 E5 D5 C5 G4 G4 E5 D5 C5 A4 A4  "
    "A4 F5 E5 D5 G5 G5 G5 G5 A5 G5 F5 D5 C5 G5  "
    # estribillo (primera vez)
    "E5 E5 E5 E5 E5 E5 E5 G5 C5 D5 E5  "
    "F5 F5 F5 F5 F5 E5 E5 E5 E5 E5 D5 D5 E5 D5 G5  "
    # estribillo (final)
    "E5 E5 E5 E5 E5 E5 E5 G5 C5 D5 E5  "
    "F5 F5 F5 F5 F5 E5 E5 E5 E5 G5 G5 F5 D5 C5"
)
ALEGRIA = (
    "E4 E4 F4 G4 G4 F4 E4 D4 C4 C4 D4 E4 E4 D4 D4  "
    "E4 E4 F4 G4 G4 F4 E4 D4 C4 C4 D4 E4 D4 C4 C4  "
    "D4 D4 E4 C4 D4 E4 F4 E4 C4 D4 E4 F4 E4 D4 C4 D4 G3  "
    "E4 E4 F4 G4 G4 F4 E4 D4 C4 C4 D4 E4 D4 C4 C4"
)
ESTRELLITA = (
    "C4 C4 G4 G4 A4 A4 G4  F4 F4 E4 E4 D4 D4 C4  "
    "G4 G4 F4 F4 E4 E4 D4  G4 G4 F4 F4 E4 E4 D4  "
    "C4 C4 G4 G4 A4 A4 G4  F4 F4 E4 E4 D4 D4 C4"
)
CUMPLE = ("G4 G4 A4 G4 C5 B4  G4 G4 A4 G4 D5 C5  "
          "G4 G4 G5 E5 C5 B4 A4  F5 F5 E5 C5 D5 C5")
MARY = ("E4 D4 C4 D4 E4 E4 E4  D4 D4 D4  E4 G4 G4  "
        "E4 D4 C4 D4 E4 E4 E4 E4  D4 D4 E4 D4 C4")

CANCIONES = [
    ("Jingle Bells", JINGLE_BELLS),
    ("Himno a la Alegria", ALEGRIA),
    ("Estrellita", ESTRELLITA),
    ("Cumpleanos feliz", CUMPLE),
    ("Mary tenia un corderito", MARY),
]

COLOR_MANO = {"L": (255, 170, 0), "R": (0, 140, 255)}   # BGR
NOMBRE_MANO = {"L": "izq", "R": "der"}
FUENTE = cv2.FONT_HERSHEY_SIMPLEX

## 3. Notas y sonido
Cada nota es un numero (0 = Do4). El sonido se sintetiza con numpy, no necesitas archivos de audio. `calcular_inicio` reparte automaticamente las notas de cada cancion entre los 10 dedos.

In [3]:
# ------------------------------ NOTAS Y SONIDO ----------------------------
# Una nota se guarda como un entero: 0 = Do4, 1 = Re4, ... 7 = Do5, -1 = Si3
def parsear(texto_notas):
    return [(int(t[1:]) - 4) * 7 + LETRAS.index(t[0]) for t in texto_notas.split()]


def nombre_nota(i):
    return f"{NOMBRES[i % 7]}{4 + i // 7}"


def frecuencia(i):
    semitonos = SEMITONOS[i % 7] + 12 * (i // 7)
    return 261.6256 * 2 ** (semitonos / 12)


def calcular_inicio(notas):
    """Nota del menique izquierdo: centra la cancion sobre los 10 dedos."""
    minimo, maximo = min(notas), max(notas)
    ancho = maximo - minimo + 1
    if ancho > 10:
        raise ValueError("La cancion abarca mas de 10 notas: no cabe en 10 dedos")
    return minimo - (10 - ancho) // 2


def indice_nota(lado, dedo, inicio):
    """Nota que toca un dedo. Izq: menique..pulgar = 0..4; Der: pulgar..menique = 5..9."""
    k = (5 - dedo) if lado == "L" else (4 + dedo)
    return inicio + k


def dedo_de_nota(nota, inicio):
    k = nota - inicio
    return ("L", 5 - k) if k < 5 else ("R", k - 4)


def sintetizar(freq, dur=1.4, sr=44100):
    """Tono tipo piano: armonicos + ataque rapido + caida exponencial."""
    t = np.linspace(0, dur, int(sr * dur), endpoint=False)
    onda = (np.sin(2 * np.pi * freq * t)
            + 0.50 * np.sin(2 * np.pi * 2 * freq * t)
            + 0.25 * np.sin(2 * np.pi * 3 * freq * t)
            + 0.12 * np.sin(2 * np.pi * 4 * freq * t))
    onda *= np.exp(-3.0 * t) * np.minimum(1.0, t / 0.005)
    onda /= np.max(np.abs(onda))
    mono = (onda * 0.6 * 32767).astype(np.int16)
    estereo = np.ascontiguousarray(np.column_stack([mono, mono]))
    return pygame.sndarray.make_sound(estereo)


def cargar_sonidos():
    return {i: sintetizar(frecuencia(i)) for i in range(-10, 21)}

## 4. Deteccion de dedos bajados
A partir de los landmarks de MediaPipe calculamos la **flexion** de cada dedo (cociente de distancias). Si cae por debajo de un umbral, el dedo esta "abajo". `asignar_manos` decide cual es la mano izquierda y cual la derecha.

In [4]:
# ------------------------------ DETECCION --------------------------------
def medir(mano, w, h):
    """Flexion de cada dedo (grande = abierto, pequeno = doblado).

    Dedos 2-5: distancia(punta, muneca) / distancia(nudillo, muneca).
    Pulgar:    distancia(punta, nudillo del menique) / ancho de la palma.
    Son cocientes: no dependen del tamano de la mano ni de la camara.
    """
    p = np.array([[l.x * w, l.y * h, l.z * w] for l in mano.landmark])

    def d(a, b):
        return float(np.linalg.norm(p[a] - p[b]))

    razones = {1: d(4, 17) / max(d(5, 17), 1e-6)}
    for dedo, (punta, nudillo) in {2: (8, 5), 3: (12, 9),
                                   4: (16, 13), 5: (20, 17)}.items():
        razones[dedo] = d(punta, 0) / max(d(nudillo, 0), 1e-6)
    return razones


def asignar_manos(res):
    """{'L': mano izquierda, 'R': mano derecha} (imagen en espejo).

    Con 2 manos se decide por la posicion; con 1 mano, por la etiqueta del modelo.
    """
    manos = res.multi_hand_landmarks or []
    if len(manos) >= 2:
        orden = sorted(manos[:2], key=lambda m: m.landmark[0].x)
        return {"L": orden[0], "R": orden[1]}
    if len(manos) == 1:
        etiqueta = res.multi_handedness[0].classification[0].label
        return {"L" if etiqueta == "Left" else "R": manos[0]}
    return {}


def nuevo_estado(razones, base):
    """Estado de una mano recien aparecida (no dispara notas por si sola)."""
    abajo = {d: r < FACTOR_ABAJO * base[d] for d, r in razones.items()}
    return {"suave": dict(razones),
            "armado": {d: not a for d, a in abajo.items()},
            "abajo": abajo,
            "ultimo": {d: 0.0 for d in razones}}


def actualizar(est, razones, base, ahora):
    """Suaviza la flexion y devuelve los dedos que acaban de bajar."""
    nuevos = []
    for dedo, r in razones.items():
        s = SUAVIZADO * est["suave"][dedo] + (1 - SUAVIZADO) * r
        est["suave"][dedo] = s
        if s < FACTOR_ABAJO * base[dedo]:
            if est["armado"][dedo] and ahora - est["ultimo"][dedo] > COOLDOWN:
                est["armado"][dedo] = False
                est["ultimo"][dedo] = ahora
                nuevos.append(dedo)
            est["abajo"][dedo] = True
        elif s > FACTOR_ARRIBA * base[dedo]:
            est["armado"][dedo] = True
            est["abajo"][dedo] = False
    return nuevos

## 5. Dibujo en pantalla y celebracion

In [5]:
# ------------------------------- DIBUJO ----------------------------------
def texto(frame, msg, pos, escala=0.7, color=(255, 255, 255), grosor=2):
    cv2.putText(frame, msg, pos, FUENTE, escala, (0, 0, 0), grosor + 2)
    cv2.putText(frame, msg, pos, FUENTE, escala, color, grosor)


def dibujar_mano(frame, mano, lado, est, objetivo, inicio, errores, ahora):
    h, w = frame.shape[:2]
    dib = mp.solutions.drawing_utils
    color_mano = COLOR_MANO[lado]
    dib.draw_landmarks(
        frame, mano, mp.solutions.hands.HAND_CONNECTIONS,
        dib.DrawingSpec(color=color_mano, thickness=2, circle_radius=2),
        dib.DrawingSpec(color=color_mano, thickness=2))
    pulso = int(22 + 4 * np.sin(ahora * 8))
    for dedo, (nombre, punta) in DEDOS.items():
        lm = mano.landmark[punta]
        x, y = int(lm.x * w), int(lm.y * h)
        abajo = est["abajo"].get(dedo, False)
        cv2.circle(frame, (x, y), 12, (0, 200, 255) if abajo else (255, 255, 255), -1)
        nota = indice_nota(lado, dedo, inicio)
        texto(frame, nombre_nota(nota), (x - 16, y - 16), 0.45, (255, 255, 255), 1)

        if ahora - errores.get((lado, dedo), -9) < 0.3:
            cv2.circle(frame, (x, y), 22, (0, 0, 255), 3)
        if objetivo == (lado, dedo):
            if est["armado"].get(dedo, True):
                cv2.circle(frame, (x, y), pulso, (0, 255, 0), 3)
                texto(frame, f"BAJA {nombre}", (x - 34, y - 42), 0.42, (0, 255, 0), 1)
            else:
                cv2.circle(frame, (x, y), pulso, (0, 255, 255), 3)
                texto(frame, "Levanta", (x - 24, y - 42), 0.42, (0, 255, 255), 1)


def dibujar_cancion(frame, nombre, notas, paso, inicio):
    h, w = frame.shape[:2]
    n = len(notas)
    texto(frame, f"{nombre} - nota {min(paso + 1, n)}/{n}", (12, 28), 0.7)
    x, y0 = 12, 42
    for i in range(paso, min(paso + 8, n)):
        lado, dedo = dedo_de_nota(notas[i], inicio)
        actual = i == paso
        tam = 92 if actual else 66
        color = COLOR_MANO[lado]
        cv2.rectangle(frame, (x, y0), (x + tam, y0 + tam), (30, 30, 30), -1)
        cv2.rectangle(frame, (x, y0), (x + tam, y0 + tam),
                      (0, 255, 0) if actual else color, 4 if actual else 2)
        texto(frame, nombre_nota(notas[i]), (x + 6, y0 + tam // 2),
              0.8 if actual else 0.55, color)
        texto(frame, f"{NOMBRE_MANO[lado]} {dedo}", (x + 6, y0 + tam - 10),
              0.55 if actual else 0.45, color, 1)
        x += tam + 8
    cv2.rectangle(frame, (0, h - 8), (int(w * paso / n), h), (0, 255, 0), -1)


COLORES_CONFETI = [(0, 0, 255), (0, 200, 255), (0, 255, 0), (255, 170, 0),
                   (255, 0, 255), (255, 255, 0), (0, 140, 255)]


def nueva_sesion():
    return {"paso": 0, "fallos": 0, "t0": None, "t1": None,
            "confeti": [], "fanfarria": [], "fiesta_hasta": 0.0}


def crear_confeti(w, n=150):
    return [{"x": random.uniform(0, w), "y": random.uniform(-300, 0),
             "vx": random.uniform(-40, 40), "vy": random.uniform(90, 230),
             "tam": random.randint(6, 11), "fase": random.uniform(0, 6.28),
             "redondo": random.random() < 0.4,
             "color": random.choice(COLORES_CONFETI)} for _ in range(n)]


def texto_centrado(frame, msg, y, escala=0.8, color=(255, 255, 255), grosor=2):
    (tw, _), _ = cv2.getTextSize(msg, FUENTE, escala, grosor)
    texto(frame, msg, ((frame.shape[1] - tw) // 2, y), escala, color, grosor)


def dibujar_celebracion(frame, nombre, ses, ahora, dt):
    """Confeti + banner con tiempo y fallos al terminar una cancion."""
    h, w = frame.shape[:2]
    y0 = h // 3
    capa = frame.copy()
    cv2.rectangle(capa, (0, y0 - 70), (w, y0 + 85), (0, 0, 0), -1)
    cv2.addWeighted(capa, 0.45, frame, 0.55, 0, frame)

    reciclar = ahora < ses["fiesta_hasta"]
    for p in ses["confeti"]:
        p["y"] += p["vy"] * dt
        p["x"] += (p["vx"] + 30 * np.sin(ahora * 3 + p["fase"])) * dt
        if reciclar and p["y"] > h + 10:
            p["y"], p["x"] = random.uniform(-40, 0), random.uniform(0, w)
        x, y, t = int(p["x"]), int(p["y"]), p["tam"]
        if p["redondo"]:
            cv2.circle(frame, (x, y), t // 2 + 1, p["color"], -1)
        else:
            cv2.rectangle(frame, (x, y), (x + t, y + t // 2 + 2), p["color"], -1)
    ses["confeti"] = [p for p in ses["confeti"] if p["y"] <= h + 10]

    color = COLORES_CONFETI[int(ahora * 4) % len(COLORES_CONFETI)]
    escala = 1.3 + 0.1 * np.sin(ahora * 6)
    texto_centrado(frame, "Cancion completada!", y0 - 10, escala, color, 3)
    texto_centrado(frame, nombre, y0 + 25, 0.8, (255, 255, 255), 2)
    if ses["t0"] is not None and ses["t1"] is not None:
        seg = int(ses["t1"] - ses["t0"])
        resumen = f"Tiempo {seg // 60}:{seg % 60:02d}   |   Fallos: {ses['fallos']}"
        texto_centrado(frame, resumen, y0 + 58, 0.65, (0, 255, 255), 2)
    texto_centrado(frame, "R = repetir    1-5 = otra cancion", y0 + 80, 0.45,
                   (255, 255, 255), 1)

## 6. Programa principal

In [6]:
# -------------------------------- MAIN -----------------------------------
def main():
    libro = []
    for nombre, texto_notas in CANCIONES:
        notas = parsear(texto_notas)
        libro.append((nombre, notas, calcular_inicio(notas)))

    pygame.mixer.pre_init(44100, -16, 2, 256)
    pygame.mixer.init()
    pygame.mixer.set_num_channels(32)
    sonidos = cargar_sonidos()

    cap = cv2.VideoCapture(CAMARA)
    if not cap.isOpened():
        raise SystemExit("No se pudo abrir la camara. Prueba CAMARA = 1.")
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 960)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 540)

    hands = mp.solutions.hands.Hands(
        max_num_hands=2, model_complexity=1,
        min_detection_confidence=0.6, min_tracking_confidence=0.5)

    base = {"L": dict(BASE_ABIERTA), "R": dict(BASE_ABIERTA)}
    est = {"L": None, "R": None}
    errores = {}
    libre, debug = False, False
    idx, inicio_libre = 0, 0
    ses = nueva_sesion()
    mensaje, mensaje_hasta = "", 0.0
    t_prev = time.time()

    print("Listo. 1-5 canciones, L libre, C calibrar, R reiniciar, Q salir.")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)               # espejo: izquierda = tu izquierda
        h, w = frame.shape[:2]
        res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ahora = time.time()
        dt = min(ahora - t_prev, 0.1)
        t_prev = ahora
        manos = asignar_manos(res)

        nombre, notas, inicio_cancion = libro[idx]
        inicio = inicio_libre if libre else inicio_cancion

        # --- eventos: dedos que acaban de bajar ---
        for lado in ("L", "R"):
            if lado not in manos:
                est[lado] = None
                continue
            razones = medir(manos[lado], w, h)
            if est[lado] is None:
                est[lado] = nuevo_estado(razones, base[lado])
            for dedo in actualizar(est[lado], razones, base[lado], ahora):
                nota = indice_nota(lado, dedo, inicio)
                if libre:
                    sonidos[nota].play()
                elif ses["paso"] < len(notas):
                    if nota == notas[ses["paso"]]:
                        sonidos[nota].play()
                        if ses["paso"] == 0:
                            ses["t0"] = ahora
                        ses["paso"] += 1
                        if ses["paso"] >= len(notas):      # cancion terminada
                            ses["t1"] = ahora
                            ses["confeti"] = crear_confeti(w)
                            ses["fiesta_hasta"] = ahora + 6
                            ses["fanfarria"] = [(ahora + 0.4 + 0.12 * k, n)
                                                for k, n in enumerate([0, 2, 4, 7, 9, 11])]
                            ses["fanfarria"] += [(ahora + 1.2, n) for n in (7, 9, 11)]
                    else:
                        errores[(lado, dedo)] = ahora
                        ses["fallos"] += 1

        pendientes = []
        for t_nota, nota in ses["fanfarria"]:
            if ahora >= t_nota:
                sonidos[nota].play()
            else:
                pendientes.append((t_nota, nota))
        ses["fanfarria"] = pendientes

        # --- dibujo ---
        objetivo = None
        if not libre and ses["paso"] < len(notas):
            objetivo = dedo_de_nota(notas[ses["paso"]], inicio)
        for lado, mano in manos.items():
            dibujar_mano(frame, mano, lado, est[lado], objetivo, inicio, errores, ahora)

        if libre:
            texto(frame, "MODO LIBRE: cada dedo toca su nota", (12, 28), 0.8, (0, 255, 255))
            texto(frame, f"Rango: {nombre_nota(inicio)} a {nombre_nota(inicio + 9)}"
                         "   ( , / . cambian la octava )", (12, 56), 0.55)
        else:
            dibujar_cancion(frame, nombre, notas, ses["paso"], inicio)
            if ses["paso"] >= len(notas):
                dibujar_celebracion(frame, nombre, ses, ahora, dt)

        if not manos:
            texto_centrado(frame, "Muestra tus manos (palmas al frente)",
                           h // 2 + 50, 0.8, (0, 200, 255))
        elif objetivo and objetivo[0] not in manos:
            lado_txt = "IZQUIERDA" if objetivo[0] == "L" else "DERECHA"
            texto_centrado(frame, f"Muestra tu mano {lado_txt}",
                           h // 2 + 50, 0.8, (0, 200, 255))

        if debug:
            y = 120
            for lado in ("L", "R"):
                if est[lado]:
                    for dedo, (nom, _) in DEDOS.items():
                        s = est[lado]["suave"][dedo]
                        umbral = FACTOR_ABAJO * base[lado][dedo]
                        texto(frame, f"{NOMBRE_MANO[lado]} {nom}: {s:.2f} (baja < {umbral:.2f})",
                              (w - 340, y), 0.5, (255, 255, 0), 1)
                        y += 22
        if ahora < mensaje_hasta:
            texto(frame, mensaje, (12, h - 70), 0.8, (0, 255, 255))
        ayuda1 = "  ".join(f"{i + 1} {c[0].split()[0]}" for i, c in enumerate(CANCIONES))
        texto(frame, ayuda1 + "  L libre", (12, h - 40), 0.5, (255, 255, 255), 1)
        texto(frame, "C calibrar  R reiniciar  D datos  Q salir", (12, h - 18),
              0.5, (255, 255, 255), 1)

        cv2.imshow("Piano con las manos", frame)
        tecla = cv2.waitKey(1) & 0xFF
        if tecla in (ord("q"), 27):
            break
        elif ord("1") <= tecla < ord("1") + len(CANCIONES):
            idx, ses, libre = tecla - ord("1"), nueva_sesion(), False
        elif tecla == ord("l"):
            libre = not libre
        elif tecla == ord("r"):
            ses = nueva_sesion()
        elif tecla == ord("d"):
            debug = not debug
        elif tecla == ord(",") and libre:
            inicio_libre = max(-7, inicio_libre - 7)
        elif tecla == ord(".") and libre:
            inicio_libre = min(7, inicio_libre + 7)
        elif tecla == ord("c"):
            visibles = [lado for lado in ("L", "R") if est[lado]]
            for lado in visibles:
                base[lado] = dict(est[lado]["suave"])
            mensaje = ("Calibrado con tus manos abiertas" if visibles
                       else "Muestra las manos abiertas y pulsa C")
            mensaje_hasta = ahora + 2.5

    cap.release()
    hands.close()
    cv2.destroyAllWindows()
    pygame.quit()

## 7. Ejecutar
Se abre una ventana aparte con la camara. Muestra las dos manos **abiertas** y pulsa `C` para calibrar. Cierra con `Q` (con la ventana de la camara seleccionada).

In [9]:
try:
    main()
finally:
    cv2.destroyAllWindows()
    cv2.waitKey(1)

Listo. 1-5 canciones, L libre, C calibrar, R reiniciar, Q salir.
